# Solutions · 05 · Analysis of rate data

Worked solutions to the exercises of [notebook 05](../notebooks/05_analysis_of_rate_data.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engreact import datasets, kinetics, rate_data
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Fit the N2O5 data with the mis-read sample included, using nonlinear regression, and compare the fitted k and its standard error with the fit excluding it. Does the standard error 'notice' the outlier? What would a robust loss do?

In [2]:
d = pd.read_csv(datasets.path("n2o5_decomposition.csv"), comment="#")
t, C = d.t_s.to_numpy(), d.C_mmol_L.to_numpy()
with_bad = rate_data.nonlinear_fit(t, C, order=1)
mask = np.ones_like(t, bool); mask[7] = False
without = rate_data.nonlinear_fit(t[mask], C[mask], order=1)
print(f"with the bad sample:    k = {with_bad['k']:.3e} +/- {with_bad['se_k']:.1e} 1/s")
print(f"without:                k = {without['k']:.3e} +/- {without['se_k']:.1e} 1/s")
print(f"the standard error with the outlier is {with_bad['se_k']/without['se_k']:.0f} x larger: the residual it leaves inflates the error estimate, which is how least squares 'notices'; a robust (soft-L1) loss would instead down-weight it")

with the bad sample:    k = 5.968e-04 +/- 1.8e-05 1/s
without:                k = 6.161e-04 +/- 4.2e-06 1/s
the standard error with the outlier is 4 x larger: the residual it leaves inflates the error estimate, which is how least squares 'notices'; a robust (soft-L1) loss would instead down-weight it


Least squares cannot exclude the bad point but its standard error grows several-fold, which is the signal to
look at the residuals. A robust loss would reduce the outlier's pull automatically - useful when you cannot
inspect every sample, dangerous when the "outlier" is real chemistry.

## Exercise 2

The Arrhenius parameters table lists A and Ea for cyclopropane isomerisation. Compute k at 450, 500 and 550 degC, the half-life at each, and the temperature at which the half-life is one minute.

In [3]:
tab = pd.read_csv(datasets.path("arrhenius_parameters.csv"), comment="#")
row = tab[tab.reaction.str.startswith("cyclopropane")].iloc[0]
A, Ea = float(row.A), row.Ea_kJ_mol * 1e3
for T_C in (450, 500, 550):
    k = kinetics.arrhenius(A, Ea, T_C + 273.15)
    print(f"{T_C} degC: k = {k:.3e} 1/s, half-life {np.log(2)/k:.1f} s")
T_1min = kinetics.temperature_for_rate_ratio(Ea, 450 + 273.15, np.log(2) / 60 / kinetics.arrhenius(A, Ea, 450 + 273.15))
print(f"half-life of one minute at {T_1min - 273.15:.0f} degC")

450 degC: k = 3.609e-05 1/s, half-life 19206.7 s
500 degC: k = 6.729e-04 1/s, half-life 1030.1 s
550 degC: k = 8.793e-03 1/s, half-life 78.8 s
half-life of one minute at 556 degC


The isomerisation of cyclopropane, the textbook unimolecular reaction, has a half-life of five hours at
450 degC and under two minutes at 550 degC, its activation energy of 272 kJ/mol making it exquisitely
temperature-sensitive; a one-minute half-life is reached near 556 degC.

## Exercise 3

**Implement it yourself:** write the half-life method (a straight line of ln t_1/2 against ln C0) and apply it to half-lives you compute from the saponification fit at four starting concentrations.

In [4]:
s = pd.read_csv(datasets.path("saponification.csv"), comment="#")
fit = rate_data.nonlinear_fit(s.t_s, s.C_mol_L, order=2)
C0s = np.array([0.01, 0.02, 0.04, 0.08])
halves = [rate_data.half_life(c, fit["k"], 2) for c in C0s]
x, y = np.log(C0s), np.log(halves)
slope = np.polyfit(x, y, 1)[0]
print(f"half-lives {np.round(halves, 1)} s; slope of ln t1/2 vs ln C0 = {slope:.3f} -> order {1 - slope:.2f}; library: {rate_data.half_life_method(C0s, halves)['order']:.2f}")

half-lives [912.9 456.4 228.2 114.1] s; slope of ln t1/2 vs ln C0 = -1.000 -> order 2.00; library: 2.00


For second order the half-life is inversely proportional to the initial concentration - slope -1, order 2 -
which is why a single half-life means nothing for a reaction of unknown order.